# Section 0 - Setup


In [ ]:
# Install dependencies
!pip install -q xgboost lightgbm imbalanced-learn optuna scikit-learn mlflow joblib shap

import numpy as np
import pandas as pd
import json
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

ML_MODELS_DIR = Path('../ml_models')
ML_MODELS_DIR.mkdir(parents=True, exist_ok=True)


# Section 1 - Load Dataset


In [ ]:
# Download Kaggle disease-symptom dataset
!kaggle datasets download -d itachi9604/disease-symptom-description-dataset -p /tmp/dataset --unzip

df = pd.read_csv('/tmp/dataset/dataset.csv')
print(f'Dataset shape: {df.shape}')
print(df.head())
print('\nColumns:', df.columns.tolist())
print('\nDisease distribution:')
print(df['Disease'].value_counts())


# Section 2 - Feature Engineering


In [ ]:
np.random.seed(42)

# Synthesize demographic and history features
n_samples = len(df)
df['age'] = np.random.normal(45, 15, n_samples).clip(18, 90)
df['age_bin'] = pd.cut(df['age'], bins=[0, 30, 50, 70, 100], labels=[0, 1, 2, 3]).astype(int)
df['gender_encoded'] = np.random.choice([0, 1], n_samples)
df['bmi'] = np.random.normal(25, 5, n_samples).clip(15, 45)
df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]).astype(int)

df['has_diabetes'] = np.random.choice([0, 1], n_samples, p=[0.9, 0.1])
df['has_hypertension'] = np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
df['has_respiratory_condition'] = np.random.choice([0, 1], n_samples, p=[0.9, 0.1])
df['allergy_count'] = np.random.poisson(0.5, n_samples)
df['medication_count'] = np.random.poisson(1, n_samples)

df['smoking_encoded'] = np.random.choice([0, 1, 2], n_samples, p=[0.7, 0.2, 0.1])
df['pack_years'] = np.where(df['smoking_encoded'] > 0, np.random.exponential(10, n_samples), 0)
df['pack_years_binned'] = pd.cut(df['pack_years'], bins=[-1, 0, 10, 20, 100], labels=[0, 1, 2, 3]).astype(int)
df['alcohol_risk'] = np.random.choice([0, 1, 2], n_samples, p=[0.6, 0.3, 0.1])
df['activity_score'] = np.random.choice([0, 1, 2], n_samples)
df['sleep_deficit'] = np.random.choice([0, 1], n_samples, p=[0.7, 0.3])

# Direct symptoms - assuming dataset has Symptom_1, Symptom_2 etc.
symptom_cols = [c for c in df.columns if c.startswith('Symptom_')]
for c in symptom_cols:
    df[c] = df[c].str.strip().str.replace(' ', '_')

# Melt and get dummies for symptoms
symptoms_df = df[symptom_cols].apply(lambda x: pd.Series(x.dropna().unique()), axis=1)
symptoms_one_hot = pd.get_dummies(symptoms_df.stack()).groupby(level=0).max()
df = pd.concat([df.drop(columns=symptom_cols), symptoms_one_hot], axis=1).fillna(0)

# Engineered Risk Multipliers
def safe_col(col):
    return df[col] if col in df.columns else 0

df['cardiac_risk_score'] = (df['age'] > 50).astype(int) * (df['smoking_encoded'] > 0).astype(int) * ((df['has_hypertension'] == 1) | (df['has_diabetes'] == 1)).astype(int)
df['respiratory_risk_score'] = (df['smoking_encoded'] > 0).astype(int) * ((safe_col('chest_pain') == 1) | (safe_col('cough') == 1) | (safe_col('breathlessness') == 1)).astype(int)
df['acute_abdomen_flag'] = safe_col('abdominal_pain') * safe_col('fever') * ((safe_col('nausea') == 1) | (safe_col('vomiting') == 1)).astype(int)
df['neurological_flag'] = (safe_col('headache') == 1) | ((safe_col('dizziness') == 1) & (safe_col('vision_changes') == 1)).astype(int)
df['infection_cluster'] = safe_col('fever') * ((safe_col('fatigue') == 1) | (safe_col('body_aches') == 1)).astype(int)

X = df.drop(columns=['Disease'])
print('Features shape:', X.shape)


# Section 3 - SMOTE + Train/Val/Test Split


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

le = LabelEncoder()
y = le.fit_transform(df['Disease'])
print(f'Number of disease classes: {len(le.classes_)}')

with open(ML_MODELS_DIR / 'label_encoder.json', 'w') as f:
    json.dump(le.classes_.tolist(), f)

# Ensure columns are numeric
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f'After SMOTE: {X_train_resampled.shape}')


# Section 4 - XGBoost baseline training + MLflow


In [ ]:
import xgboost as xgb
import mlflow
from sklearn.metrics import f1_score

mlflow.set_experiment('xgboost_symptom_classifier')

with mlflow.start_run(run_name='baseline'):
    model = xgb.XGBClassifier(
        n_estimators=100,
        eval_metric='mlogloss',
        early_stopping_rounds=20,
        random_state=42
    )
    model.fit(X_train_resampled, y_train_resampled, eval_set=[(X_val, y_val)], verbose=False)
    
    preds = model.predict(X_val)
    val_f1 = f1_score(y_val, preds, average='macro')
    mlflow.log_metric('val_macro_f1', val_f1)
    print(f'Baseline Val Macro-F1: {val_f1:.4f}')


# Section 5 - Optuna Hyperparameter Search


In [ ]:
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'eval_metric': 'mlogloss',
        'early_stopping_rounds': 20,
        'random_state': 42
    }
    
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        model = xgb.XGBClassifier(**params)
        model.fit(X_train_resampled, y_train_resampled, eval_set=[(X_val, y_val)], verbose=False)
        
        preds = model.predict(X_val)
        f1 = f1_score(y_val, preds, average='macro')
        mlflow.log_metric('val_macro_f1', f1)
        return f1

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, n_jobs=1)
print('Best trial:', study.best_trial.params)


# Section 6 - Best model evaluation


In [ ]:
from sklearn.metrics import confusion_matrix
import shap

best_params = study.best_trial.params
best_params['eval_metric'] = 'mlogloss'
best_params['random_state'] = 42

X_train_val = pd.concat([X_train_resampled, X_val])
y_train_val = np.concatenate([y_train_resampled, y_val])

best_model = xgb.XGBClassifier(**best_params)
best_model.fit(X_train_val, y_train_val, verbose=False)

test_preds = best_model.predict(X_test)
test_f1 = f1_score(y_test, test_preds, average='macro')
print(f'Test Macro-F1: {test_f1:.4f}')

cm = confusion_matrix(y_test, test_preds, normalize='true')
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues')
plt.title('Normalized Confusion Matrix')
plt.tight_layout()
plt.show()

f1_scores = f1_score(y_test, test_preds, average=None)
plt.figure(figsize=(10, 6))
plt.bar(range(len(f1_scores)), f1_scores)
plt.title('Per-class F1 Score')
plt.tight_layout()
plt.show()

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, max_display=20, plot_type='bar')


# Section 7 - Ablation: with vs without engineered features


In [ ]:
engineered_cols = ['cardiac_risk_score', 'respiratory_risk_score', 'acute_abdomen_flag', 'neurological_flag', 'infection_cluster']
X_train_abl = X_train_val.drop(columns=engineered_cols, errors='ignore')
X_test_abl = X_test.drop(columns=engineered_cols, errors='ignore')

model_abl = xgb.XGBClassifier(**best_params)
model_abl.fit(X_train_abl, y_train_val, verbose=False)
preds_abl = model_abl.predict(X_test_abl)
f1_abl = f1_score(y_test, preds_abl, average='macro')

print(f'Test Macro-F1 (Full features): {test_f1:.4f}')
print(f'Test Macro-F1 (Without engineered): {f1_abl:.4f}')
print(f'Delta: {test_f1 - f1_abl:.4f}')


# Section 8 - Save


In [ ]:
joblib.dump(best_model, ML_MODELS_DIR / 'xgboost_symptom.pkl')
with open(ML_MODELS_DIR / 'feature_names.json', 'w') as f:
    json.dump(X.columns.tolist(), f)
print('Model saved ✓')
